In [77]:
import numpy as np
import torch


In [3]:
x = torch.tensor([1, 2, 3])
torch.is_tensor(x)
# Returns True if obj is a PyTorch tensor.

True

In [7]:
torch.is_floating_point(torch.tensor([1.0, 2.0, 3.0]))

True

In [8]:
torch.is_floating_point(torch.tensor([1, 2, 3], dtype=torch.int32))

False

In [9]:
torch.is_floating_point(torch.tensor([1.0, 2.0, 3.0], dtype=torch.float16))

True

In [10]:
torch.is_nonzero(torch.tensor([0.]))

False

In [18]:
torch.set_default_dtype(torch.float64)
torch.tensor([1.2, 3]).dtype

torch.float64

In [19]:
torch.set_default_dtype(torch.float16)
torch.tensor([1.2, 3]).dtype

torch.float16

In [20]:
torch.get_default_dtype()

torch.float16

In [30]:
torch.set_default_device('cpu')
torch.get_default_device()

device(type='cpu')

In [27]:
torch.set_default_device('cuda')  # current device is 0
#torch.get_default_device()

In [32]:
# Returns the total number of elements in the input tensor
a = torch.randn(1, 2, 3, 4, 5)
torch.numel(a)
# 1 * 2 * 3 * 4 * 5

120

In [33]:
a = torch.zeros(4,4)
torch.numel(a)

16

In [53]:
# Limit the precision of elements
torch.set_printoptions(precision=3)
torch.tensor([1.12345])
# Restore defaults
# torch.set_printoptions(profile='default')
# torch.tensor([1.12345])

tensor([1.123])

In [46]:
# Limit the number of elements shown
# If a tensor has more than 5 elements, don't show me the whole thing.
torch.set_printoptions(threshold=5)
torch.arange(10)

tensor([0, 1, 2,  ..., 7, 8, 9])

In [54]:
torch.tensor([[0.1, 1.2], [2.2, 3.1], [4.9, 5.2]])
# torch.tensor([[0.11111, 0.222222, 0.3333333]],
#              dtype=torch.float64,
#              device=torch.device('cuda:0'))

tensor([[0.100, 1.200],
        [2.199, 3.100],
        [4.898, 5.199]])

In [57]:
torch.tensor(3.14159)  # Create a zero-dimensional (scalar) tensor

tensor(3.141)

In [58]:
torch.tensor([])  # Create an empty tensor (of size (0,))

tensor([])

In [62]:
i = torch.tensor([[0, 1, 1],[2, 0, 2]])  # (0, 2) (1,0) (1,2)
v = torch.tensor([3, 4, 5], dtype=torch.float32)
torch.sparse_coo_tensor(i, v, [2, 4])
# torch.sparse_coo_tensor(i, v)  # Shape inference
# torch.sparse_coo_tensor(i, v, [2, 4],
#                         dtype=torch.float64,
#                         device=torch.device('cuda:0'))

tensor(indices=tensor([[0, 1, 1],
                       [2, 0, 2]]),
       values=tensor([3., 4., 5.]),
       size=(2, 3), nnz=3, dtype=torch.float32, layout=torch.sparse_coo)

In [66]:
# Compressed Sparse Row
crow_indices = [0, 2, 4]
col_indices = [0, 1, 0, 1]
values = [1, 2, 3, 4]
torch.sparse_csr_tensor(torch.tensor(crow_indices, dtype=torch.int64),
                        torch.tensor(col_indices, dtype=torch.int64),
                        torch.tensor(values), dtype=torch.double)

/usr/local/lib/python3.12/dist-packages/torch/utils/_device.py:109: UserWarning: Sparse CSR tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at /pytorch/aten/src/ATen/SparseCsrTensorImpl.cpp:49.)
  return func(*args, **kwargs)


tensor(crow_indices=tensor([0, 2, 4]),
       col_indices=tensor([0, 1, 0, 1]),
       values=tensor([1., 2., 3., 4.]), size=(2, 2), nnz=4,
       dtype=torch.float64, layout=torch.sparse_csr)

In [ ]:
# Compressed Sparse Col
ccol_indices = [0, 2, 4]
row_indices = [0, 1, 0, 1]
values = [1, 2, 3, 4]
torch.sparse_csc_tensor(torch.tensor(ccol_indices, dtype=torch.int64),
                        torch.tensor(row_indices, dtype=torch.int64),
                        torch.tensor(values), dtype=torch.double)

In [72]:
# for converting data into tensors. Its primary superpower is efficiency—it
# tries to avoid copying memory whenever possible, unlike torch.tensor(), which almost always creates a fresh copy.

# When you use torch.asarray(a), PyTorch looks at the source.
# Since a is already a tensor with the same properties, b simply points to the exact same spot in your RAM.

a = torch.tensor([1, 2, 3])
b = torch.asarray(a)
a.data_ptr() == b.data_ptr()  # True - They are two names for the same data - shared memory

c = torch.asarray(a, copy=True)
a.data_ptr() == c.data_ptr()  # False - This forces PyTorch to clone the data into a new memory address. Now, changing a will not affect c.


a = torch.tensor([1., 2., 3.], requires_grad=True)
b = a + 2
b

# Default (c = torch.asarray(b)):
# It creates a "view" that points to the data but detaches it from the gradient history.
# c.requires_grad will be False. This is useful if you want to use the values for something like logging without affecting backpropagation.

# Retaining History (d = torch.asarray(b, requires_grad=True)):
# By explicitly setting the flag, d remains connected to the graph.
# It still shares memory with b, but it "remembers" how it was calculated.


tensor([1., 2., 3.], requires_grad=True)

In [75]:

array = np.array([1, 2, 3])
# Shares memory with array 'array'
t1 = torch.asarray(array)
array.__array_interface__['data'][0] == t1.data_ptr()  # True

# Direct Link: When the dtype matches (e.g., both are int64), t1 shares the exact memory buffer of the NumPy array.
# No extra RAM is used.

# Copies memory due to dtype mismatch
t2 = torch.asarray(array, dtype=torch.float32)
array.__array_interface__['data'][0] == t2.data_ptr()  # False

# The Dtype Trigger: As soon as you request a different type (dtype=torch.float32),
# PyTorch must create a copy because it has to translate the bit representation of the numbers.

scalar = np.float64(0.5)
torch.asarray(scalar)

tensor(0.500, dtype=torch.float64)

In [80]:
# zero_copy
# PyTorch simply points to the exact same memory address that NumPy is using. There is no data movement;
# they are two different "views" of the same physical bits in your RAM.

a = np.array([1, 2, 3])
t = torch.as_tensor(a)
print(t)
t[0] = -1
print(a)

tensor([1, 2, 3])
[-1  2  3]


In [ ]:
# Your NumPy array a lives in System RAM (CPU memory). Your GPU has its own dedicated memory (VRAM).
# To create a tensor on the GPU from a CPU array,
# PyTorch must copy the data across the PCIe bus from the RAM to the VRAM.

a = np.array([1, 2, 3])
t = torch.as_tensor(a, device=torch.device('cuda'))
print(t)

t[0] = -1
print(a)